In [ ]:
from ultralytics import YOLO
model = YOLO("./models/base/yolo26n-seg.pt")

In [ ]:
import albumentations as A

IMG_SIZE = 1088
IMG_SIZE = 1088

# Aggressive augmentations layered on top of Ultralytics' built-in geometric/color jitter.
# Targets robustness to phone-camera artefacts: tight/partial crops, sensor noise, JPEG compression.
augmentations = [
    # A.RandomSizedBBoxSafeCrop(height=IMG_SIZE, width=IMG_SIZE, erosion_rate=0.2, p=0.3),
    # A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(0.05, 0.2), hole_width_range=(0.05, 0.2), p=0.3),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.3),
    A.ISONoise(p=0.2),
    A.ImageCompression(quality_range=(40, 90), p=0.3),
    A.Blur(blur_limit=5, p=0.1),
    A.MedianBlur(blur_limit=5, p=0.05),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomGamma(p=0.2),
    A.ToGray(p=0.02),
    A.CLAHE(p=0.05),
]

model.train(data="./datasets/window_segmentation/window_segmentation.yaml", 
    epochs=150, 
    imgsz=IMG_SIZE,
    single_cls=True,
    degrees=10.0,    # Slightly rotate images
    flipud=0.5,      # Flip up-down
    fliplr=0.5,      # Flip left-right
    mosaic=1.0,       # Combine multiple images into one
    scale=0.9,
    shear=5.0,
    perspective=0.0005,
    mixup=0.15,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    augmentations=augmentations,
    )

In [ ]:
model.save("./models/tuned/window_segmentation_yolo26n-seg.pt")

In [ ]:
import glob
import os
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO("./models/tuned/window_segmentation_yolo26n-seg.pt")
print("loaded model")
test_images_path = "./datasets/window_segmentation/test/images/*.jpg"  # Update path if needed
image_paths = glob.glob(test_images_path)

if not image_paths:
    raise FileNotFoundError("No images found. Please verify your test images path.")

# Select a random subset to display (e.g., 4 samples)
num_samples = min(4, len(image_paths))
sampled_paths = random.sample(image_paths, num_samples)

# 3. Setup Matplotlib grid (assuming a 2x2 grid for 4 samples)
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.ravel()  # Flatten the 2D array of axes into 1D for easy iteration

# 4. Infer and plot
for idx, img_path in enumerate(sampled_paths):
    # Run segmentation prediction
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    
    # Extract the first result from the list
    result = results[0]
    

    annotated_img = result.plot(pil=True, boxes=True, masks=True)
    
    axes[idx].imshow(annotated_img)
    axes[idx].set_title(os.path.basename(img_path), fontsize=10)
    axes[idx].axis("off")  # Hide grid borders and coordinates

# Tight layout adjustments for notebook embedding
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def iterate_for_4_points(mask_contour, max_iterations=100, tolerance=1e-5):
    """
    Iteratively adjusts epsilon to force cv2.approxPolyDP to return exactly 4 points.
    """
    perimeter = cv2.arcLength(mask_contour, True)
    if perimeter == 0:
        return None

    # Define binary search bounds for epsilon factor
    low = 0.0
    high = 1.0
    
    best_approx = None
    best_distance_to_four = float('inf')

    for _ in range(max_iterations):
        mid = (low + high) / 2.0
        epsilon = mid * perimeter
        approx = cv2.approxPolyDP(mask_contour, epsilon, True)
        num_points = len(approx)

        if num_points == 4:
            return approx.reshape(4, 2)
        
        # Track the closest approximation just in case exactly 4 is mathematically impossible
        distance = abs(num_points - 4)
        if distance < best_distance_to_four:
            best_distance_to_four = distance
            best_approx = approx

        # If it has too many points, epsilon is too small -> increase lower bound
        if num_points > 4:
            low = mid
        # If it has too few points, epsilon is too large -> decrease upper bound
        else:
            high = mid

        if (high - low) < tolerance:
            break

    # Fallback: If 4 points couldn't be strictly resolved, return the closest match
    if best_approx is not None:
        return best_approx.reshape(-1, 2)
    
    return None

def reduce_to_4_points_by_area(points):
    """
    If an approximation yields > 4 points, greedily removes the point 
    that minimizes the loss of polygon area until 4 points remain.
    """
    pts = list(points)
    while len(pts) > 4:
        best_idx = -1
        max_area = -1
        
        # Test the remaining area if we drop point `i`
        for i in range(len(pts)):
            test_pts = pts[:i] + pts[i+1:]
            area = cv2.contourArea(np.array(test_pts, dtype=np.int32))
            if area > max_area:
                max_area = area
                best_idx = i
                
        pts.pop(best_idx)
        
    return np.array(pts, dtype=np.int32)


# Extract your contour from YOLO's mask.xy

def to4PointPoly(mask):
    contour = np.int32([mask])

    points = iterate_for_4_points(contour)

    if len(points) > 4:
        points = reduce_to_4_points_by_area(points)
    elif len(points) < 4:
        hull = cv2.convexHull(contour)
        points = reduce_to_4_points_by_area(hull.reshape(-1, 2))

    return points


model = YOLO("./models/tuned/window_segmentation_yolo26n-seg.pt")
print("loaded model")
test_images_path = "./datasets/window_segmentation/test/images/*.jpg" 
image_paths = glob.glob(test_images_path)

if not image_paths:
    raise FileNotFoundError("No images found. Please verify your test images path.")

num_samples = min(4, len(image_paths))
sampled_paths = random.sample(image_paths, num_samples)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.ravel()

for idx, img_path in enumerate(sampled_paths):
    # Load original image and convert BGR to RGB for matplotlib display
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run prediction
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    result = results[0]
    
    # Display the clean base image
    axes[idx].imshow(img)
    
    # Check if any masks were detected
    if result.masks is not None:
        masks_xy = result.masks.xy  # Get boundary coordinates for each mask
        
        for mask in masks_xy:
            # Generate the fitted 4-point polygon coordinates
            poly_points = to4PointPoly(mask)
            
            if len(poly_points) == 4:
                # Create a Matplotlib Polygon patch for clean overlay rendering
                polygon_patch = patches.Polygon(
                    poly_points, 
                    closed=True, 
                    edgecolor='#00FF00', # Bright green border
                    facecolor='#00FF00', # Semi-transparent green fill
                    alpha=0.3, 
                    linewidth=2.5
                )
                axes[idx].add_patch(polygon_patch)
                
                # Optional: Plot markers on the 4 corner points to verify placement
                axes[idx].scatter(poly_points[:, 0], poly_points[:, 1], color='red', s=40, zorder=3)

    axes[idx].set_title(os.path.basename(img_path), fontsize=10)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()